<a href="https://www.kaggle.com/code/amryasswe/complete-ml-pipeline-tutorial-2026?scriptVersionId=320848565" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# %% [code]
# %% [markdown]
# # Complete ML Pipeline: From Data to Deployment (2026)
# 
# A production-ready Machine Learning pipeline tutorial covering every step:
# 1. Data Loading & EDA
# 2. Feature Engineering
# 3. Model Training & Comparison
# 4. Hyperparameter Tuning
# 5. Model Evaluation
# 6. Feature Importance & Explainability
# 
# **Dataset**: We'll predict employee attrition using synthetic HR data.

# %% [code]
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print("=" * 60)
print("  COMPLETE ML PIPELINE TUTORIAL")
print("=" * 60)

# %% [markdown]
# ## Step 1: Generate & Load Data

# %% [code]
# Generate realistic HR dataset
np.random.seed(42)
n = 2000

data = {
    'age': np.random.randint(22, 62, n),
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR', 'Finance', 'Operations'], n),
    'salary': np.random.randint(30000, 200000, n),
    'years_at_company': np.random.randint(0, 30, n),
    'satisfaction_score': np.round(np.random.uniform(1, 10, n), 1),
    'work_life_balance': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.15, 0.35, 0.30, 0.15]),
    'num_projects': np.random.randint(1, 10, n),
    'avg_monthly_hours': np.random.randint(120, 300, n),
    'promotion_last_5years': np.random.choice([0, 1], n, p=[0.75, 0.25]),
    'performance_rating': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.40, 0.30, 0.15]),
    'distance_from_home': np.random.randint(1, 50, n),
    'education_level': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n, p=[0.15, 0.45, 0.30, 0.10]),
    'overtime': np.random.choice([0, 1], n, p=[0.6, 0.4]),
}

df = pd.DataFrame(data)

# Create target with realistic correlations
attrition_prob = (
    0.1 +
    (df['satisfaction_score'] < 4).astype(float) * 0.25 +
    (df['avg_monthly_hours'] > 250).astype(float) * 0.15 +
    (df['overtime'] == 1).astype(float) * 0.1 +
    (df['years_at_company'] < 2).astype(float) * 0.1 +
    (df['salary'] < 50000).astype(float) * 0.1 -
    (df['promotion_last_5years'] == 1).astype(float) * 0.1
)
attrition_prob = np.clip(attrition_prob, 0.05, 0.85)
df['attrition'] = (np.random.random(n) < attrition_prob).astype(int)

print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Distribution:")
print(f"  No Attrition (0): {(df['attrition']==0).sum()} ({(df['attrition']==0).mean()*100:.1f}%)")
print(f"  Attrition (1):    {(df['attrition']==1).sum()} ({(df['attrition']==1).mean()*100:.1f}%)")
print(f"\nFirst 5 rows:")
print(df.head())

# %% [markdown]
# ## Step 2: Exploratory Data Analysis

# %% [code]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Age distribution by attrition
for att, color, label in [(0, '#4ecdc4', 'Stayed'), (1, '#ff6b6b', 'Left')]:
    axes[0,0].hist(df[df['attrition']==att]['age'], bins=20, alpha=0.6, color=color, label=label)
axes[0,0].set_title('Age Distribution by Attrition', fontweight='bold')
axes[0,0].legend()

# 2. Salary by attrition
df.boxplot(column='salary', by='attrition', ax=axes[0,1])
axes[0,1].set_title('Salary by Attrition', fontweight='bold')
axes[0,1].set_xlabel('Attrition')

# 3. Satisfaction by attrition
df.boxplot(column='satisfaction_score', by='attrition', ax=axes[0,2])
axes[0,2].set_title('Satisfaction by Attrition', fontweight='bold')

# 4. Department attrition rate
dept_attr = df.groupby('department')['attrition'].mean().sort_values()
axes[1,0].barh(dept_attr.index, dept_attr.values * 100, color='#667eea', alpha=0.8)
axes[1,0].set_title('Attrition Rate by Department (%)', fontweight='bold')

# 5. Monthly hours
for att, color, label in [(0, '#4ecdc4', 'Stayed'), (1, '#ff6b6b', 'Left')]:
    axes[1,1].hist(df[df['attrition']==att]['avg_monthly_hours'], bins=20, alpha=0.6, color=color, label=label)
axes[1,1].set_title('Monthly Hours by Attrition', fontweight='bold')
axes[1,1].legend()

# 6. Correlation heatmap (numeric only)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
im = axes[1,2].imshow(corr.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1,2].set_xticks(range(len(numeric_cols)))
axes[1,2].set_yticks(range(len(numeric_cols)))
axes[1,2].set_xticklabels(numeric_cols, rotation=90, fontsize=7)
axes[1,2].set_yticklabels(numeric_cols, fontsize=7)
axes[1,2].set_title('Correlation Matrix', fontweight='bold')
plt.colorbar(im, ax=axes[1,2], fraction=0.046)

plt.suptitle('Employee Attrition - EDA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## Step 3: Feature Engineering & Preprocessing

# %% [code]
# Encode categoricals
le_dept = LabelEncoder()
le_edu = LabelEncoder()
df['department_encoded'] = le_dept.fit_transform(df['department'])
df['education_encoded'] = le_edu.fit_transform(df['education_level'])

# Feature engineering
df['salary_per_hour'] = df['salary'] / (df['avg_monthly_hours'] * 12)
df['years_per_project'] = df['years_at_company'] / (df['num_projects'] + 1)
df['overworked'] = ((df['avg_monthly_hours'] > 200) & (df['overtime'] == 1)).astype(int)

# Select features
feature_cols = ['age', 'salary', 'years_at_company', 'satisfaction_score', 
                'work_life_balance', 'num_projects', 'avg_monthly_hours',
                'promotion_last_5years', 'performance_rating', 'distance_from_home',
                'overtime', 'department_encoded', 'education_encoded',
                'salary_per_hour', 'years_per_project', 'overworked']

X = df[feature_cols]
y = df['attrition']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Features:     {len(feature_cols)}")

# %% [markdown]
# ## Step 4: Model Training & Comparison

# %% [code]
# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

results = {}
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 75)

for name, model in models.items():
    # Use scaled data for LR and SVM, original for tree-based
    if name in ['Logistic Regression', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc,
                     'y_pred': y_pred, 'y_proba': y_proba}
    
    print(f"{name:<25} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc:>10.4f}")

# Cross-validation
print(f"\n{'Model':<25} {'CV Mean':>10} {'CV Std':>10}")
print("-" * 45)
for name, model in models.items():
    data = X_train_scaled if name in ['Logistic Regression', 'SVM'] else X_train
    cv_scores = cross_val_score(model, data, y_train, cv=5, scoring='f1')
    print(f"{name:<25} {cv_scores.mean():>10.4f} {cv_scores.std():>10.4f}")

# %% [markdown]
# ## Step 5: Model Evaluation & Visualization

# %% [code]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0,0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)
axes[0,0].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[0,0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('False Positive Rate')
axes[0,0].set_ylabel('True Positive Rate')
axes[0,0].legend(fontsize=9)

# 2. Model Comparison
metrics_df = pd.DataFrame({name: {k: v for k, v in res.items() if k not in ['y_pred', 'y_proba']} 
                           for name, res in results.items()}).T
metrics_df.plot(kind='bar', ax=axes[0,1], colormap='viridis', alpha=0.8)
axes[0,1].set_title('Model Comparison', fontsize=14, fontweight='bold')
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=30, ha='right')
axes[0,1].legend(fontsize=8)
axes[0,1].set_ylim(0, 1)

# 3. Best model confusion matrix
best_model_name = max(results, key=lambda x: results[x]['f1'])
cm = confusion_matrix(y_test, results[best_model_name]['y_pred'])
im = axes[1,0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[1,0].text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=16, fontweight='bold')
axes[1,0].set_title(f'Confusion Matrix ({best_model_name})', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Predicted')
axes[1,0].set_ylabel('Actual')
axes[1,0].set_xticks([0,1])
axes[1,0].set_yticks([0,1])
axes[1,0].set_xticklabels(['Stayed', 'Left'])
axes[1,0].set_yticklabels(['Stayed', 'Left'])

# 4. Feature Importance (Random Forest)
rf = models['Random Forest']
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True).tail(10)
feat_imp.plot(kind='barh', ax=axes[1,1], color='#667eea', alpha=0.85)
axes[1,1].set_title('Top 10 Feature Importances (RF)', fontsize=14, fontweight='bold')

plt.suptitle('ML Model Evaluation Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest Model: {best_model_name} (F1={results[best_model_name]['f1']:.4f})")

# %% [markdown]
# ## Step 6: Hyperparameter Tuning

# %% [code]
# GridSearchCV on the best model
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Final evaluation
y_pred_final = grid_search.predict(X_test)
y_proba_final = grid_search.predict_proba(X_test)[:, 1]

print(f"\nFinal Model Performance:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_test, y_proba_final):.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=['Stayed', 'Left']))

# %% [markdown]
# ## Key Takeaways
# 
# 1. **Satisfaction score** is the strongest predictor of employee attrition
# 2. **Monthly hours** and **overtime** are significant risk factors
# 3. **Gradient Boosting** and **Random Forest** outperform linear models
# 4. **Feature engineering** (overworked flag, salary per hour) improved performance
# 
# **If this pipeline tutorial was useful, please upvote!**
